# Archaeological Site Extraction with Google Earth Engine

This notebook extracts archaeological site information from PDFs using LLM and retrieves remote sensing data from Google Earth Engine.

**Workflow:**
1. Configure API keys and parameters
2. Install dependencies
3. Initialize Google Earth Engine
4. Upload and extract text from PDF
5. Use LLM to extract site coordinates
6. Download remote sensing data from GEE
7. Visualize and package data

## 1. Configuration

**Edit these parameters before running:**

In [ ]:
# ============================================================================
# CONFIGURATION - EDIT THESE PARAMETERS
# ============================================================================

# OpenAI API Key (required for LLM extraction)
OPENAI_API_KEY = "your-api-key-here"

# Google Earth Engine Project ID (required)
# Get this from: https://console.cloud.google.com/
GEE_PROJECT_ID = "your-project-id-here"

# OpenAI Model to use
OPENAI_MODEL = "gpt-4o-mini"  # Options: gpt-4o, gpt-4-turbo, gpt-3.5-turbo

# GEE Extraction Parameters
CELL_SIZE_KM = 1.0  # Size of extraction area (km)
PIXELS_PER_KM = 100  # Resolution (100 = 100x100 pixels per km²)

# Sentinel-2 Parameters
DATE_START = '2020-01-01'
DATE_END = '2024-12-31'
CLOUD_COVER_MAX = 20  # Maximum cloud cover percentage

# Output Options
SAVE_TO_DRIVE = False  # Save outputs to Google Drive?

print("✓ Configuration loaded")

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q pypdf openai earthengine-api

print("✓ Dependencies installed")

## 3. Initialize Google Earth Engine

**This will open a browser window for authentication:**

In [ ]:
import ee

# Authenticate (opens browser window)
try:
    ee.Authenticate()
    print("✓ Authentication successful")
except Exception as e:
    print(f"Authentication error: {e}")
    print("Please follow the instructions above to authenticate.")

# Initialize with your project ID
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"✓ Google Earth Engine initialized with project: {GEE_PROJECT_ID}")

    # Test the connection
    test_image = ee.Image('USGS/SRTMGL1_003')
    print("✓ GEE connection verified")
except Exception as e:
    print(f"❌ Initialization error: {e}")
    print("Make sure your GEE_PROJECT_ID is correct.")

## 4. Import Libraries and Define Helper Functions

In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile
import zipfile
from google.colab import files
from openai import OpenAI
import pypdf
from scipy.ndimage import zoom

# Initialize OpenAI client
import os
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
client = OpenAI()

print("✓ Libraries imported")

### Coordinate Parsing Functions

In [ ]:
def parse_coordinate_string(coord_str):
    """
    Parse coordinate string in various formats to decimal degrees.

    Handles formats like:
    - "5.23°S, 60.12°W"
    - "5°13'48\"S, 60°7'12\"W"
    - "9°57′38.96″S, 67°29′51.39″W" (Unicode primes)
    - "-5.23, -60.12"

    Returns: (lat, lon) as floats, or (None, None) if parsing fails
    """
    if not coord_str:
        return None, None

    coord_str = coord_str.strip()
    parts = [p.strip() for p in coord_str.split(',')]
    if len(parts) != 2:
        return None, None

    try:
        lat = parse_single_coordinate(parts[0])
        lon = parse_single_coordinate(parts[1])

        if lat is not None and lon is not None:
            return float(lat), float(lon)
    except Exception as e:
        print(f"  Error parsing coordinates: {e}")
        pass

    return None, None


def parse_single_coordinate(coord_str):
    """
    Parse a single coordinate value.
    Returns: float or None
    """
    coord_str = coord_str.strip()

    # Check for direction suffix (N/S/E/W)
    direction = 1
    if coord_str and coord_str[-1].upper() in ['S', 'W']:
        direction = -1
        coord_str = coord_str[:-1].strip()
    elif coord_str and coord_str[-1].upper() in ['N', 'E']:
        direction = 1
        coord_str = coord_str[:-1].strip()

    # Remove degree symbol at the end
    coord_str = coord_str.rstrip('°')

    # Try DMS format with multiple possible symbols
    # Handles: ° (degree), ' ′ (minutes - apostrophe or prime), " ″ (seconds - quote or double prime)
    # Unicode characters: ′ is prime (U+2032), ″ is double prime (U+2033)
    import re
    dms_pattern = r"(-?\d+)[°\s]+(\d+(?:\.\d+)?)[′'\s]+(\d+(?:\.\d+)?)[″\"]?"
    match = re.match(dms_pattern, coord_str)

    if match:
        degrees = float(match.group(1))
        minutes = float(match.group(2))
        seconds = float(match.group(3))

        decimal = abs(degrees) + minutes/60 + seconds/3600
        if degrees < 0:
            decimal = -decimal

        return float(decimal * direction)

    # Try decimal format
    try:
        decimal = float(coord_str)
        return float(decimal * direction)
    except ValueError:
        return None

print("✓ Coordinate parsing functions defined")

## 5. Upload PDF File

In [ ]:
# Upload PDF file
print("Please upload a PDF file:")
uploaded = files.upload()

# Get the uploaded filename
pdf_filename = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {pdf_filename}")

## 6. Extract Text from PDF

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file"""
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = pypdf.PdfReader(file)
        num_pages = len(reader.pages)

        print(f"Extracting text from {num_pages} pages...")
        for i, page in enumerate(reader.pages):
            page_text = page.extract_text() or ""
            text += f"\n--- Page {i+1} ---\n"
            text += page_text
            print(f"  Page {i+1}/{num_pages}", end='\r')

    print(f"\n✓ Extracted {len(text)} characters")
    return text

# Extract text
paper_text = extract_text_from_pdf(pdf_filename)

# Preview first 500 characters
print("\nText preview:")
print("=" * 60)
print(paper_text[:500])
print("...")
print("=" * 60)

## 7. Extract Site Information Using LLM

In [ ]:
def create_extraction_prompt(paper_text):
    """Create the extraction prompt"""

    prompt = """# Archaeological Site Information Extraction

    ## Task
    Extract information about archaeological sites from the provided academic paper. Your role is to act as a precise data extractor, NOT an interpreter or estimator.

    ## CRITICAL RULES - READ CAREFULLY

    1. **ONLY extract information explicitly stated in the paper**
      - If coordinates are not given, leave the coordinates field empty
      - If a site name is not mentioned, do not invent one
      - If information is ambiguous or unclear, mark it as such

    2. **NEVER:**
      - Guess or estimate coordinates from place names
      - Invent precision that doesn't exist in the source
      - Convert between coordinate systems unless explicitly shown in the paper
      - Fill in missing information based on general knowledge
      - Assume information from context alone

    3. **ALWAYS:**
      - Preserve the exact format of coordinates as written in the paper
      - Note when information is approximate, estimated, or uncertain
      - Include the specific page or section where information was found
      - Flag when coordinates are withheld or stated as "not disclosed"

    ## Information to Extract

    For each archaeological site mentioned in the paper, extract:

    ### Site Identification
    - **site_name**: The exact name(s) used in the paper
    - **site_code**: Any alphanumeric codes or identifiers
    - **alternative_names**: Other names mentioned for the same site

    ### Location Information
    - **coordinates_explicit**: ONLY if coordinates are explicitly provided
      - Extract the exact text as written (e.g., "5.23°S, 60.12°W")
      - Note the format: decimal_degrees, DMS, UTM, etc.
      - Note the datum if specified (WGS84, SAD69, etc.)
      - Mark precision level: "exact", "approximate", "rounded"

    - **location_description**: Textual descriptions of location
    - **administrative_location**: Modern political boundaries mentioned
    - **location_withheld**: Boolean - true if paper explicitly states location is not disclosed

    ### Temporal Information
    - **dating**: Dates or date ranges mentioned
    - **cultural_period**: Cultural phases or periods named
    - **dating_method**: If specified
    - **dating_uncertainty**: Any caveats about dating

    ### Site Characteristics
    - **site_type**: Settlement, mound, ceremonial center, earthwork, etc.
    - **site_features**: Specific features mentioned
    - **site_size**: Dimensions if provided
    - **site_condition**: Preservation state if mentioned

    ### Context
    - **study_type**: Is this a site being actively studied or just mentioned for comparison?
    - **source_location**: Page number(s) or section where this information appears
    - **confidence_level**: Your assessment - "high", "medium", "low"
    - **extraction_notes**: Any important caveats, ambiguities, or clarifications

    ## Output Format

    Return a JSON object with this structure:

    {
      "paper_metadata": {
        "title": "extracted from paper",
        "authors": ["list", "of", "authors"],
        "year": "publication year",
        "doi": "if available"
      },
      "extraction_summary": {
        "total_sites_found": 0,
        "sites_with_explicit_coordinates": 0,
        "sites_with_descriptions_only": 0,
        "extraction_date": "YYYY-MM-DD"
      },
      "sites": [
        {
          "site_name": "string or null",
          "site_code": "string or null",
          "alternative_names": [],
          "coordinates": {
            "has_explicit_coordinates": false,
            "raw_text": null,
            "format": null,
            "latitude": null,
            "longitude": null,
            "datum": null,
            "precision_level": null
          },
          "location_description": null,
          "administrative_location": {
            "country": null,
            "state_province": null,
            "other": null
          },
          "location_withheld": false,
          "temporal": {
            "dating": null,
            "cultural_period": null,
            "dating_method": null,
            "uncertainty": null
          },
          "characteristics": {
            "site_type": null,
            "features": [],
            "size": null,
            "condition": null
          },
          "metadata": {
            "study_type": null,
            "source_location": null,
            "confidence_level": null,
            "extraction_notes": null
          }
        }
      ]
    }

    ## Your Task

    Extract all archaeological site information from the following paper:

    """

    return prompt + "\n" + paper_text + "\n\nReturn ONLY the JSON output, no additional commentary."


def extract_sites_with_llm(paper_text):
    """Use LLM to extract site information"""

    prompt = create_extraction_prompt(paper_text)

    print("Sending to LLM for extraction...")
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=16000
    )

    response_text = response.choices[0].message.content
    print("✓ Extraction complete")

    # Parse JSON response
    try:
        # Remove markdown code blocks if present
        if response_text.startswith("```json"):
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif response_text.startswith("```"):
            response_text = response_text.split("```")[1].split("```")[0].strip()

        extracted_data = json.loads(response_text)
        return extracted_data
    except json.JSONDecodeError as e:
        return {"raw_response": response_text, "parse_error": str(e)}


# Extract sites
extracted_data = extract_sites_with_llm(paper_text)

## 8. Display Extraction Results

In [ ]:
# Display extraction summary
if "extraction_summary" in extracted_data:
    summary = extracted_data["extraction_summary"]
    print("=" * 60)
    print("EXTRACTION SUMMARY")
    print("=" * 60)
    print(f"Total sites found: {summary.get('total_sites_found', 0)}")
    print(f"Sites with coordinates: {summary.get('sites_with_explicit_coordinates', 0)}")
    print(f"Sites with descriptions only: {summary.get('sites_with_descriptions_only', 0)}")
    print()

# Display sites with coordinates
sites_with_coords = []
if "sites" in extracted_data:
    print("Sites with explicit coordinates:")
    print("=" * 60)

    for i, site in enumerate(extracted_data["sites"]):
        if site.get("coordinates", {}).get("has_explicit_coordinates"):
            sites_with_coords.append(site)
            print(f"\n[{i}] {site.get('site_name', 'Unnamed')}")
            coords = site.get('coordinates', {})
            print(f"    Raw: {coords.get('raw_text')}")
            print(f"    Lat: {coords.get('latitude')}, Lon: {coords.get('longitude')}")
            print(f"    Type: {site.get('characteristics', {}).get('site_type')}")
            print(f"    Dating: {site.get('temporal', {}).get('dating')}")

print(f"\n✓ Found {len(sites_with_coords)} sites with coordinates")

# Save extracted data
output_json = pdf_filename.replace('.pdf', '_extracted.json')
with open(output_json, 'w') as f:
    json.dump(extracted_data, f, indent=2)
print(f"✓ Saved extraction results to: {output_json}")

## 9. Google Earth Engine Data Extraction Functions

In [ ]:
def create_grid_bbox(lat, lon, cell_size_km):
    """Create bounding box for extraction"""
    half_size_deg = (cell_size_km / 2) / 111.32

    min_lon = lon - half_size_deg
    max_lon = lon + half_size_deg
    min_lat = lat - half_size_deg
    max_lat = lat + half_size_deg

    return ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])


def get_sentinel2_image(roi):
    """Get least cloudy Sentinel-2 image for region"""
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(DATE_START, DATE_END)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_COVER_MAX))
        .sort('CLOUDY_PIXEL_PERCENTAGE')
    )

    return ee.Image(collection.first())


def get_dem_data(roi):
    """Get SRTM DEM elevation data"""
    dem = ee.Image('USGS/SRTMGL1_003')
    return dem


def calculate_slope(dem_image):
    """Calculate slope from DEM"""
    return ee.Terrain.slope(dem_image)


def extract_band_array(image, band, roi, pixels):
    """Extract a single band as numpy array"""
    band_image = image.select(band)

    array = band_image.sampleRectangle(region=roi, defaultValue=0)
    data = array.get(band).getInfo()
    arr = np.array(data, dtype=np.float32)

    # Diagnostic info
    print(f"  {band}: shape={arr.shape}, min={np.min(arr):.2f}, max={np.max(arr):.2f}")

    # Resize if needed
    if arr.shape != (pixels, pixels):
        zoom_factors = (pixels / arr.shape[0], pixels / arr.shape[1])
        arr = zoom(arr, zoom_factors, order=1)
        print(f"    Resized to ({pixels}, {pixels})")

    return arr


def calculate_ndvi(b8, b4):
    """Calculate NDVI"""
    numerator = b8 - b4
    denominator = b8 + b4
    ndvi = np.divide(numerator, denominator,
                    out=np.zeros_like(numerator),
                    where=denominator!=0)
    return ndvi.astype(np.float32)


def calculate_ndwi(b3, b8):
    """Calculate NDWI"""
    numerator = b3 - b8
    denominator = b3 + b8
    ndwi = np.divide(numerator, denominator,
                    out=np.zeros_like(numerator),
                    where=denominator!=0)
    return ndwi.astype(np.float32)


def calculate_bsi(b11, b4, b8, b2):
    """Calculate BSI (Bare Soil Index)"""
    numerator = (b11 + b4) - (b8 + b2)
    denominator = (b11 + b4) + (b8 + b2)
    bsi = np.divide(numerator, denominator,
                   out=np.zeros_like(numerator),
                   where=denominator!=0)
    return bsi.astype(np.float32)


print("✓ GEE extraction functions defined")

## 10. Extract Remote Sensing Data for a Site

**Select a site to extract data for:**

In [ ]:
# Select site index (or enter coordinates manually)
SITE_INDEX = 0  # Change this to select different site

# Option 1: Use extracted site
if sites_with_coords and SITE_INDEX < len(sites_with_coords):
    selected_site = sites_with_coords[SITE_INDEX]
    site_name = selected_site.get('site_name', 'Unknown')

    coords = selected_site['coordinates']
    lat = coords.get('latitude')
    lon = coords.get('longitude')

    # Convert to float if they're strings
    if isinstance(lat, str):
        try:
            lat = float(lat)
        except:
            lat = None
    if isinstance(lon, str):
        try:
            lon = float(lon)
        except:
            lon = None

    # Try to parse if not already decimal or parsing failed
    if lat is None or lon is None:
        raw_text = coords.get('raw_text', '')
        print(f"Parsing coordinates from: {raw_text}")
        lat, lon = parse_coordinate_string(raw_text)

    print(f"Selected site: {site_name}")
    if lat is not None and lon is not None:
        print(f"Coordinates: {lat}, {lon}")
    else:
        print(f"Coordinates: Could not parse")

# Option 2: Manual coordinates (uncomment and edit if needed)
# site_name = "Manual Site"
# lat = -5.23
# lon = -60.12
# print(f"Using manual coordinates: {lat}, {lon}")

if lat is None or lon is None:
    print("❌ Could not determine coordinates. Please set them manually above.")
else:
    # Ensure they're floats
    lat = float(lat)
    lon = float(lon)
    print(f"✓ Ready to extract data for ({lat:.4f}, {lon:.4f})")

## 11. Download and Process GEE Data

In [ ]:
def extract_gee_data(lat, lon, site_name="site"):
    """
    Extract all GEE data for a location

    Returns: dict with channels and metadata
    """
    pixels = PIXELS_PER_KM

    # Create region of interest
    roi = create_grid_bbox(lat, lon, CELL_SIZE_KM)

    print("Getting imagery...")
    # Get imagery
    s2_image = get_sentinel2_image(roi)
    dem_image = get_dem_data(roi)
    slope_image = calculate_slope(dem_image)

    # Extract all bands
    channels = {}

    print("\nExtracting Sentinel-2 bands...")
    bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
    for band in bands:
        channels[band] = extract_band_array(s2_image, band, roi, pixels)

    print("\nExtracting DEM...")
    channels['DEM'] = extract_band_array(dem_image, 'elevation', roi, pixels)

    print("\nExtracting Slope...")
    channels['Slope'] = extract_band_array(slope_image, 'slope', roi, pixels)

    # Calculate indices
    print("\nCalculating spectral indices...")
    channels['NDVI'] = calculate_ndvi(channels['B8'], channels['B4'])
    channels['NDWI'] = calculate_ndwi(channels['B3'], channels['B8'])
    channels['BSI'] = calculate_bsi(channels['B11'], channels['B4'],
                                     channels['B8'], channels['B2'])

    print(f"  NDVI: min={np.min(channels['NDVI']):.2f}, max={np.max(channels['NDVI']):.2f}")
    print(f"  NDWI: min={np.min(channels['NDWI']):.2f}, max={np.max(channels['NDWI']):.2f}")
    print(f"  BSI: min={np.min(channels['BSI']):.2f}, max={np.max(channels['BSI']):.2f}")

    # Get metadata
    try:
        image_info = s2_image.getInfo()
        metadata = {
            'site_name': site_name,
            'latitude': lat,
            'longitude': lon,
            'cell_size_km': CELL_SIZE_KM,
            'image_id': image_info.get('id') if image_info else None,
            'cloud_cover': image_info['properties'].get('CLOUDY_PIXEL_PERCENTAGE') if image_info else None,
            'acquisition_date': image_info['properties'].get('GENERATION_TIME') if image_info else None
        }
    except:
        metadata = {
            'site_name': site_name,
            'latitude': lat,
            'longitude': lon,
            'cell_size_km': CELL_SIZE_KM
        }

    return {'channels': channels, 'metadata': metadata}


# Extract data
if lat is not None and lon is not None:
    print(f"\nExtracting GEE data for {site_name}...")
    print("=" * 60)

    gee_data = extract_gee_data(lat, lon, site_name)

    print("\n" + "=" * 60)
    print("✓ Extraction complete!")
    print(f"Extracted {len(gee_data['channels'])} channels")
else:
    print("⚠ Skipping extraction - no valid coordinates")

## 12. Visualize Extracted Data

In [ ]:
def create_overview_visualization(channels, site_name):
    """
    Create 2x3 grid visualization of all channels.

    Layout:
    [RGB Composite] [NDVI] [NDWI]
    [BSI]           [DEM]  [Slope]
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'Site: {site_name}', fontsize=16)

    # RGB Composite (B4=Red, B3=Green, B2=Blue)
    r = channels['B4']
    g = channels['B3']
    b = channels['B2']
    rgb = np.stack([r, g, b], axis=-1)

    # Normalize RGB to 0-1 range using percentile stretch
    p2, p98 = np.percentile(rgb, [2, 98])
    rgb_norm = np.clip((rgb - p2) / (p98 - p2), 0, 1)

    axes[0, 0].imshow(rgb_norm)
    axes[0, 0].set_title('RGB Composite (B4-B3-B2)')
    axes[0, 0].axis('off')

    # NDVI
    im1 = axes[0, 1].imshow(channels['NDVI'], cmap='RdYlGn', vmin=-0.5, vmax=0.8)
    axes[0, 1].set_title('NDVI')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    # NDWI
    im2 = axes[0, 2].imshow(channels['NDWI'], cmap='Blues', vmin=-0.5, vmax=0.5)
    axes[0, 2].set_title('NDWI')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)

    # BSI
    im3 = axes[1, 0].imshow(channels['BSI'], cmap='YlOrBr', vmin=-0.5, vmax=0.5)
    axes[1, 0].set_title('BSI')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046)

    # DEM
    im4 = axes[1, 1].imshow(channels['DEM'], cmap='terrain')
    axes[1, 1].set_title('DEM (Elevation)')
    axes[1, 1].axis('off')
    plt.colorbar(im4, ax=axes[1, 1], fraction=0.046)

    # Slope
    im5 = axes[1, 2].imshow(channels['Slope'], cmap='plasma')
    axes[1, 2].set_title('Slope')
    axes[1, 2].axis('off')
    plt.colorbar(im5, ax=axes[1, 2], fraction=0.046)

    plt.tight_layout()
    plt.show()


# Visualize
if 'gee_data' in globals():
    create_overview_visualization(gee_data['channels'], site_name)
else:
    print("No data to visualize")

## 13. Save and Download Data

In [ ]:
def save_data_as_zip(data, site_name, output_path):
    """
    Package extracted data as a ZIP file.

    Structure:
    site_name/
        channels/
            B2.npy, B3.npy, ..., NDVI.npy, etc.
        metadata.json
    """
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Save each channel as .npy
        for channel_name, array in data['channels'].items():
            # Save to temporary file
            temp_npy = tempfile.NamedTemporaryFile(delete=False, suffix='.npy')
            np.save(temp_npy.name, array)
            temp_npy.close()

            # Add to zip
            zipf.write(temp_npy.name,
                      f"{site_name}/channels/{channel_name}.npy")

            # Clean up temp file
            os.unlink(temp_npy.name)

        # Save metadata as JSON
        metadata_str = json.dumps(data['metadata'], indent=2)
        zipf.writestr(f"{site_name}/metadata.json", metadata_str)


if 'gee_data' in globals():
    # Create safe filename
    safe_site_name = "".join(c if c.isalnum() or c in ('-', '_') else '_'
                            for c in site_name)
    zip_filename = f"{safe_site_name}_{lat:.4f}_{lon:.4f}.zip"

    # Save data
    print(f"\nPackaging data into {zip_filename}...")
    save_data_as_zip(gee_data, safe_site_name, zip_filename)

    print("✓ Package created")
    print(f"\nDownloading {zip_filename}...")

    # Download file
    files.download(zip_filename)

    print("✓ Download complete!")
else:
    print("No data to save")

## 14. Batch Processing (Optional)

Extract data for all sites with coordinates:

In [ ]:
# Uncomment to process all sites

# BATCH_PROCESS = False  # Set to True to process all sites

# if BATCH_PROCESS and sites_with_coords:
#     print(f"\nBatch processing {len(sites_with_coords)} sites...")
#     print("=" * 60)

#     for i, site in enumerate(sites_with_coords):
#         print(f"\n[{i+1}/{len(sites_with_coords)}] Processing {site.get('site_name', 'Unknown')}...")

#         # Get coordinates
#         coords = site['coordinates']
#         lat = coords.get('latitude')
#         lon = coords.get('longitude')

#         # Convert to float if strings
#         if isinstance(lat, str):
#             try:
#                 lat = float(lat)
#             except:
#                 lat = None
#         if isinstance(lon, str):
#             try:
#                 lon = float(lon)
#             except:
#                 lon = None

#         if lat is None or lon is None:
#             raw_text = coords.get('raw_text', '')
#             lat, lon = parse_coordinate_string(raw_text)

#         if lat is None or lon is None:
#             print(f"  ⚠ Skipping - could not parse coordinates")
#             continue

#         try:
#             # Ensure floats
#             lat = float(lat)
#             lon = float(lon)

#             # Extract data
#             site_name = site.get('site_name', f'site_{i}')
#             data = extract_gee_data(lat, lon, site_name)

#             # Save
#             safe_name = "".join(c if c.isalnum() or c in ('-', '_') else '_'
#                                for c in site_name)
#             zip_filename = f"{safe_name}_{lat:.4f}_{lon:.4f}.zip"
#             save_data_as_zip(data, safe_name, zip_filename)

#             print(f"  ✓ Saved: {zip_filename}")

#         except Exception as e:
#             print(f"  ❌ Error: {e}")

#     print("\n" + "=" * 60)
#     print("✓ Batch processing complete!")

print("Batch processing cell ready (uncomment to use)")